# Wash Shuffle Analysis

Analysis of card shuffling data in `data/`. Each CSV records, per trial, where each card started and ended.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

In [ ]:
# Load all CSVs into one DataFrame
files = sorted(DATA_DIR.glob("*.csv"))
dfs = [pd.read_csv(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(df))

Files loaded: ['Caleb_wash_shuffle_data_combined.csv', 'Jordan_wash_shuffle_data.csv', 'Peter_wash_shuffle_data.csv', 'Sam_wash_shuffle_data.csv', 'Seena_wash_shuffle_data.csv']
Total rows: 45344


,name,sequence_id,trial_id,card_number,start_position,end_position
45334,Seena,OuwLxVLyXz,15,8H,43,46
45335,Seena,OuwLxVLyXz,15,4D,44,50
45336,Seena,OuwLxVLyXz,15,7H,45,26
45337,Seena,OuwLxVLyXz,15,8D,46,8
45338,Seena,OuwLxVLyXz,15,4S,47,48
45339,Seena,OuwLxVLyXz,15,3D,48,52
45340,Seena,OuwLxVLyXz,15,10D,49,41
45341,Seena,OuwLxVLyXz,15,AS,50,18
45342,Seena,OuwLxVLyXz,15,2C,51,43
45343,Seena,OuwLxVLyXz,15,9D,52,27


In [ ]:
# Number of data per person
df['name'].value_counts().reset_index()

,name,count
0,Sam,10920
1,Seena,10764
2,Caleb,10140
3,Peter,8840
4,Jordan,4680


In [ ]:
# I want to be able to just look at wash shuffles
df['unique_trial_id'] = df['sequence_id'].astype(str) + '_' + df['trial_id'].astype(str)


,name,sequence_id,trial_id,card_number,start_position,end_position,unique_trial_id
0,Caleb,4EFiQojyGF,1,4S,1,42,4EFiQojyGF_1
1,Caleb,4EFiQojyGF,1,JC,2,1,4EFiQojyGF_1
2,Caleb,4EFiQojyGF,1,9C,3,11,4EFiQojyGF_1
3,Caleb,4EFiQojyGF,1,9D,4,17,4EFiQojyGF_1
4,Caleb,4EFiQojyGF,1,8S,5,15,4EFiQojyGF_1
...,...,...,...,...,...,...,...
45339,Seena,OuwLxVLyXz,15,3D,48,52,OuwLxVLyXz_15
45340,Seena,OuwLxVLyXz,15,10D,49,41,OuwLxVLyXz_15
45341,Seena,OuwLxVLyXz,15,AS,50,18,OuwLxVLyXz_15
45342,Seena,OuwLxVLyXz,15,2C,51,43,OuwLxVLyXz_15


In [23]:
# Assigns a unique integer to each distinct (name, sequence_id, trial_id) combo
df['unique_trial_id'] = df.groupby(['name', 'sequence_id', 'trial_id']).ngroup() + 1
general_washes = df.drop(columns=['sequence_id', 'trial_id'])

In [22]:
df[df["unique_trial_id"] == 1]

,name,sequence_id,trial_id,card_number,start_position,end_position,unique_trial_id
3120,Caleb,2lHdZ3m3Fp,1,AS,1,16,1
3121,Caleb,2lHdZ3m3Fp,1,2S,2,1,1
3122,Caleb,2lHdZ3m3Fp,1,3S,3,38,1
3123,Caleb,2lHdZ3m3Fp,1,4S,4,29,1
3124,Caleb,2lHdZ3m3Fp,1,5S,5,45,1
3125,Caleb,2lHdZ3m3Fp,1,6S,6,24,1
3126,Caleb,2lHdZ3m3Fp,1,7S,7,7,1
3127,Caleb,2lHdZ3m3Fp,1,8S,8,8,1
3128,Caleb,2lHdZ3m3Fp,1,9S,9,25,1
3129,Caleb,2lHdZ3m3Fp,1,10S,10,26,1


In [24]:
trial_summary = df.groupby(['name', 'unique_trial_id']).agg({
    'card_number': list,
    'start_position': list,
    'end_position': list
}).reset_index()

trial_summary

,name,unique_trial_id,card_number,start_position,end_position
0,Caleb,1,"[AS, 2S, 3S, 4S, 5S, 6S, 7S, 8S, 9S, 10S, JS, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[16, 1, 38, 29, 45, 24, 7, 8, 25, 26, 27, 28, ..."
1,Caleb,2,"[2S, 2H, AC, 4H, 5H, 6H, 7S, 8S, 8H, 6D, 10H, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[30, 32, 20, 14, 24, 25, 51, 21, 3, 22, 49, 13..."
2,Caleb,3,"[QC, QH, 8H, JD, AS, KD, 9S, 4D, 8D, KC, 5D, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[36, 14, 8, 49, 32, 1, 25, 4, 17, 11, 20, 6, 1..."
3,Caleb,4,"[KD, JC, 10C, 4D, 6D, 10S, 7H, 8H, 10H, 8C, KC...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[8, 9, 3, 28, 16, 7, 39, 25, 13, 2, 4, 5, 6, 1..."
4,Caleb,5,"[6C, 8C, 10C, KC, 9C, 4H, 10S, KD, JC, 4S, 2D,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[17, 18, 23, 4, 3, 12, 24, 31, 20, 25, 44, 13,..."
...,...,...,...,...,...
867,Seena,868,"[9D, JS, 10S, 9S, JH, AS, 3C, JD, 9C, 5D, 2S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[2, 3, 4, 11, 12, 13, 14, 24, 27, 29, 32, 33, ..."
868,Seena,869,"[KS, 9D, JS, 10S, 2H, QC, 6H, 7C, 5S, 7D, 9S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[43, 48, 26, 13, 33, 10, 1, 2, 3, 6, 8, 11, 12..."
869,Seena,870,"[6H, 7C, 5S, 9C, 8S, 7D, 5D, 9S, 7H, QC, JH, A...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[38, 39, 40, 43, 44, 45, 20, 17, 19, 21, 22, 2..."
870,Seena,871,"[5C, 10D, QH, KH, 4D, 4S, 10H, KS, 6C, 6D, KD,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[39, 16, 4, 20, 45, 49, 10, 17, 23, 32, 33, 44..."


In [28]:
# calculate the distance card moved in each position
trial_summary['distance'] = [
    [abs(end - start) for start, end in zip(s, e)]
    for s, e in zip(trial_summary['start_position'], trial_summary['end_position'])
]
trial_summary

,name,unique_trial_id,card_number,start_position,end_position,distance
0,Caleb,1,"[AS, 2S, 3S, 4S, 5S, 6S, 7S, 8S, 9S, 10S, JS, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[16, 1, 38, 29, 45, 24, 7, 8, 25, 26, 27, 28, ...","[15, 1, 35, 25, 40, 18, 0, 0, 16, 16, 16, 16, ..."
1,Caleb,2,"[2S, 2H, AC, 4H, 5H, 6H, 7S, 8S, 8H, 6D, 10H, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[30, 32, 20, 14, 24, 25, 51, 21, 3, 22, 49, 13...","[29, 30, 17, 10, 19, 19, 44, 13, 6, 12, 38, 1,..."
2,Caleb,3,"[QC, QH, 8H, JD, AS, KD, 9S, 4D, 8D, KC, 5D, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[36, 14, 8, 49, 32, 1, 25, 4, 17, 11, 20, 6, 1...","[35, 12, 5, 45, 27, 5, 18, 4, 8, 1, 9, 6, 1, 1..."
3,Caleb,4,"[KD, JC, 10C, 4D, 6D, 10S, 7H, 8H, 10H, 8C, KC...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[8, 9, 3, 28, 16, 7, 39, 25, 13, 2, 4, 5, 6, 1...","[7, 7, 0, 24, 11, 1, 32, 17, 4, 8, 7, 7, 7, 4,..."
4,Caleb,5,"[6C, 8C, 10C, KC, 9C, 4H, 10S, KD, JC, 4S, 2D,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[17, 18, 23, 4, 3, 12, 24, 31, 20, 25, 44, 13,...","[16, 16, 20, 0, 2, 6, 17, 23, 11, 15, 33, 1, 3..."
...,...,...,...,...,...,...
867,Seena,868,"[9D, JS, 10S, 9S, JH, AS, 3C, JD, 9C, 5D, 2S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[2, 3, 4, 11, 12, 13, 14, 24, 27, 29, 32, 33, ...","[1, 1, 1, 7, 7, 7, 7, 16, 18, 19, 21, 21, 21, ..."
868,Seena,869,"[KS, 9D, JS, 10S, 2H, QC, 6H, 7C, 5S, 7D, 9S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[43, 48, 26, 13, 33, 10, 1, 2, 3, 6, 8, 11, 12...","[42, 46, 23, 9, 28, 4, 6, 6, 6, 4, 3, 1, 1, 20..."
869,Seena,870,"[6H, 7C, 5S, 9C, 8S, 7D, 5D, 9S, 7H, QC, JH, A...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[38, 39, 40, 43, 44, 45, 20, 17, 19, 21, 22, 2...","[37, 37, 37, 39, 39, 39, 13, 9, 10, 11, 11, 11..."
870,Seena,871,"[5C, 10D, QH, KH, 4D, 4S, 10H, KS, 6C, 6D, KD,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[39, 16, 4, 20, 45, 49, 10, 17, 23, 32, 33, 44...","[38, 14, 1, 16, 40, 43, 3, 9, 14, 22, 22, 32, ..."


In [30]:
trial_summary['mean_distance'] = trial_summary['distance'].apply(np.mean)
trial_summary['std_distance'] = trial_summary['distance'].apply(lambda x: np.std(x, ddof=1))

In [33]:
trial_summary.tail(10)

,name,unique_trial_id,card_number,start_position,end_position,distance,mean_distance,std_distance
862,Seena,863,"[4S, 2D, 7D, 4C, 8H, QS, KD, 10D, 9D, 4D, 9H, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[17, 16, 2, 29, 22, 14, 25, 21, 23, 32, 33, 36...","[16, 14, 1, 25, 17, 8, 18, 13, 14, 22, 22, 24,...",15.692308,9.572302
863,Seena,864,"[JD, 7D, 2S, 2H, 6H, 7C, JS, 10C, QH, 5H, 8D, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[28, 37, 38, 39, 32, 33, 40, 45, 47, 50, 51, 5...","[27, 35, 35, 35, 27, 27, 33, 37, 38, 40, 40, 4...",17.000000,10.717404
864,Seena,865,"[KC, QS, 5D, 7S, 3C, 10D, 2D, 8H, 9D, AH, KD, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[25, 29, 31, 35, 36, 3, 27, 14, 40, 45, 47, 9,...","[24, 27, 28, 31, 31, 3, 20, 6, 31, 35, 36, 3, ...",15.769231,12.227605
865,Seena,866,"[6H, 7C, 10D, JH, AS, 3D, JC, 3H, 10S, KH, 4S,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[52, 2, 4, 16, 17, 21, 25, 22, 41, 42, 43, 45,...","[51, 0, 1, 12, 12, 15, 18, 14, 32, 32, 32, 33,...",19.000000,12.016329
866,Seena,867,"[2D, 7C, 7S, 10D, 5S, 9D, 10C, 6D, 8C, 9S, 2H,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[47, 21, 24, 25, 52, 1, 44, 46, 48, 4, 49, 37,...","[46, 19, 21, 21, 47, 5, 37, 38, 39, 6, 38, 25,...",15.653846,13.162461
867,Seena,868,"[9D, JS, 10S, 9S, JH, AS, 3C, JD, 9C, 5D, 2S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[2, 3, 4, 11, 12, 13, 14, 24, 27, 29, 32, 33, ...","[1, 1, 1, 7, 7, 7, 7, 16, 18, 19, 21, 21, 21, ...",12.769231,12.174572
868,Seena,869,"[KS, 9D, JS, 10S, 2H, QC, 6H, 7C, 5S, 7D, 9S, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[43, 48, 26, 13, 33, 10, 1, 2, 3, 6, 8, 11, 12...","[42, 46, 23, 9, 28, 4, 6, 6, 6, 4, 3, 1, 1, 20...",13.846154,9.766659
869,Seena,870,"[6H, 7C, 5S, 9C, 8S, 7D, 5D, 9S, 7H, QC, JH, A...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[38, 39, 40, 43, 44, 45, 20, 17, 19, 21, 22, 2...","[37, 37, 37, 39, 39, 39, 13, 9, 10, 11, 11, 11...",22.846154,12.946927
870,Seena,871,"[5C, 10D, QH, KH, 4D, 4S, 10H, KS, 6C, 6D, KD,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[39, 16, 4, 20, 45, 49, 10, 17, 23, 32, 33, 44...","[38, 14, 1, 16, 40, 43, 3, 9, 14, 22, 22, 32, ...",18.307692,13.579885
871,Seena,872,"[QD, 8D, 10C, QH, 7C, 3D, 5S, 2H, 3C, 10H, JD,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[39, 1, 10, 2, 3, 4, 6, 7, 11, 12, 13, 14, 15,...","[38, 1, 7, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 22, 1...",11.769231,11.719960
